In [0]:
# gold.fact_inventory_full

from pyspark.sql import functions as F
from pyspark.sql.window import Window

pos = spark.read.table("mini_project_team_a3t7.silver.pos_transactions")
wh  = spark.read.table("mini_project_team_a3t7.silver.warehouse_inventory")
pm  = spark.read.table("mini_project_team_a3t7.silver.product_master")

# Grain: store_id + product_id + snapshot_date
pos_agg = (
    pos
    .withColumn("snapshot_date",
                F.to_date(F.col("event_timestamp").cast("timestamp")))
    .withColumn("quantity",     F.col("quantity").cast("int"))
    .withColumn("unit_price",   F.col("unit_price").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .groupBy("store_id", "product_id", "snapshot_date")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("unit_price").alias("avg_selling_price"),
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        # Channel split
        F.sum(F.when(F.col("channel") == "online",  F.col("quantity")).otherwise(0))
         .alias("units_sold_online"),
        F.sum(F.when(F.col("channel") == "offline", F.col("quantity")).otherwise(0))
         .alias("units_sold_offline"),
        # Payment method split
        F.sum(F.when(F.col("payment_method") == "cash",           F.col("total_amount")).otherwise(0))
         .alias("revenue_cash"),
        F.sum(F.when(F.col("payment_method") == "credit_card",    F.col("total_amount")).otherwise(0))
         .alias("revenue_credit_card"),
        F.sum(F.when(F.col("payment_method") == "debit_card",     F.col("total_amount")).otherwise(0))
         .alias("revenue_debit_card"),
        F.sum(F.when(F.col("payment_method") == "mobile_payment", F.col("total_amount")).otherwise(0))
         .alias("revenue_mobile_payment"),
    )
)

wh_agg = (
    wh
    .withColumn("current_stock_qty",   F.col("current_stock_qty").cast("int"))
    .withColumn("reserved_stock_qty",  F.col("reserved_stock_qty").cast("int"))
    .withColumn("available_stock_qty", F.col("available_stock_qty").cast("int"))
    .withColumn("reorder_level",       F.col("reorder_level").cast("int"))
    .withColumn("max_stock",           F.col("max_stock").cast("int"))
    .groupBy("product_id")
    .agg(
        F.sum("current_stock_qty").alias("current_stock_qty"),
        F.sum("reserved_stock_qty").alias("reserved_stock_qty"),
        F.sum("available_stock_qty").alias("available_stock_qty"),
        F.max("reorder_level").alias("reorder_level"),
        F.sum("max_stock").alias("max_stock"),
        F.first("location_zone").alias("primary_location_zone"),
        F.countDistinct("warehouse_id").alias("warehouse_count"),
        F.collect_set("warehouse_id").alias("warehouse_ids"),
        F.max("last_updated").alias("last_stock_updated"),
    )
)

pm_window = Window.partitionBy("product_id").orderBy(F.col("effective_date").desc())

pm_dedup = (
    pm
    .withColumn("cost_price",    F.col("cost_price").cast("double"))
    .withColumn("selling_price", F.col("selling_price").cast("double"))
    .withColumn("weight",        F.col("weight").cast("double"))
    .withColumn("rn", F.row_number().over(pm_window))
    .filter(F.col("rn") == 1)
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "supplier_id",
        "cost_price",
        "selling_price",
        "weight",
        "status",
        F.col("effective_date").alias("product_effective_date"),
    )
)

w30_cover = (
    Window.partitionBy("store_id", "product_id")
          .orderBy("snapshot_date")
          .rowsBetween(-30, 0)
)

# ── Join (unchanged) ──────────────────────────────────────────────────────────
fact_inv_raw = (
    pos_agg
    .join(F.broadcast(pm_dedup), on="product_id", how="inner")
    .join(F.broadcast(wh_agg),   on="product_id", how="inner")
)
 
# ── Derived columns ───────────────────────────────────────────────────────────
fact_inv = (
    fact_inv_raw
 
    # ── Profitability (unchanged) ──────────────────────────────────────────
    .withColumn(
        "gross_profit",
        F.round(
            (F.col("avg_selling_price") - F.col("cost_price")) * F.col("units_sold"),
            2
        )
    )
    .withColumn(
        "gross_margin_pct",
        F.round(
            F.when(
                F.col("total_revenue") > 0,
                (F.col("total_revenue") - F.col("cost_price") * F.col("units_sold"))
                / F.col("total_revenue") * 100
            ).otherwise(0.0),
            2
        )
    )
    .withColumn(
        "profit_per_unit",
        F.round(F.col("avg_selling_price") - F.col("cost_price"), 2)
    )
 
    # ── FIX: Compute 30-day rolling avg daily sales BEFORE stock_cover_days ──
    .withColumn(
        "avg_daily_sales_30d",
        F.round(F.avg("units_sold").over(w30_cover), 2)
    )
 
    # ── FIX: stock_cover_days now uses avg_daily_sales_30d as denominator ─────
  
    .withColumn(
        "stock_cover_days",
        F.round(
            F.when(
                F.col("avg_daily_sales_30d") > 0,       # was: units_sold > 0
                F.col("available_stock_qty") / F.col("avg_daily_sales_30d")
            ).otherwise(F.lit(None).cast("double")),
            1
        )
    )
 
    # ── Inventory health flags (unchanged logic, correct denominator now) ──
    .withColumn(
        "reorder_flag",
        F.when(F.col("available_stock_qty") <= F.col("reorder_level"),
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "stockout_flag",
        F.when(F.col("available_stock_qty") == 0,
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "overstock_flag",
        F.when(F.col("current_stock_qty") > F.col("max_stock") * 0.9,
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "sell_through_rate",
        F.round(
            F.when(
                (F.col("units_sold") + F.col("available_stock_qty")) > 0,
                F.col("units_sold") / (F.col("units_sold") + F.col("available_stock_qty")) * 100
            ).otherwise(0.0),
            2
        )
    )
    .withColumn(
        "stock_utilisation_pct",
        F.round(
            F.when(F.col("max_stock") > 0,
                   F.col("current_stock_qty") / F.col("max_stock") * 100
            ).otherwise(0.0),
            1
        )
    )
    .withColumn(
        "online_sales_pct",
        F.round(
            F.when(F.col("units_sold") > 0,
                   F.col("units_sold_online") / F.col("units_sold") * 100
            ).otherwise(0.0),
            1
        )
    )
 
    # ── Surrogate key (unchanged) ──────────────────────────────────────────
    .withColumn(
        "inventory_snapshot_key",
        F.sha2(
            F.concat_ws("|",
                F.col("store_id"),
                F.col("product_id"),
                F.col("snapshot_date").cast("string")
            ),
            256
        )
    )
    .withColumn("dw_created_at", F.current_timestamp())
)
 
# ── Final select (avg_daily_sales_30d added to output) ───────────────────────
fact_inv_final = fact_inv.select(
    "inventory_snapshot_key",
    "snapshot_date",
    "store_id",
    "product_id",
    "supplier_id",
    "product_name",
    "category",
    "subcategory",
    "brand",
    "status",
    "cost_price",
    "selling_price",
    "units_sold",
    "avg_daily_sales_30d",       
    "total_revenue",
    "avg_selling_price",
    "transaction_count",
    "unique_customers",
    "units_sold_online",
    "units_sold_offline",
    "revenue_cash",
    "revenue_credit_card",
    "revenue_debit_card",
    "revenue_mobile_payment",
    "gross_profit",
    "gross_margin_pct",
    "profit_per_unit",
    "current_stock_qty",
    "reserved_stock_qty",
    "available_stock_qty",
    "reorder_level",
    "max_stock",
    "warehouse_count",
    "primary_location_zone",
    "warehouse_ids",
    "last_stock_updated",
    "stock_cover_days",          
    "sell_through_rate",
    "stock_utilisation_pct",
    "online_sales_pct",
    "reorder_flag",
    "stockout_flag",
    "overstock_flag",
    "dw_created_at",
)

print("Writing gold.fact_inventory_full ...")

(
    fact_inv_final
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")
    .saveAsTable("mini_project_team_a3t7.gold.fact_inventory_full")
)

print("✅ gold.fact_inventory_full done.")

In [0]:
# gold.daily_inventory_fact

from pyspark.sql.functions import (
    col, sum, count, avg, round, when,
    lit, greatest, coalesce, date_trunc
)
from delta.tables import DeltaTable

gold_table = "mini_project_team_a3t7.gold.daily_inventory_fact"

# ── Step 1: Read Streaming POS Data ───────────────────────

df_pos = (
    spark.readStream
    .table("mini_project_team_a3t7.silver.pos_transactions")
)

# Static dimension tables
df_wh = spark.table("mini_project_team_a3t7.silver.warehouse_inventory")
df_sinv = spark.table("mini_project_team_a3t7.silver.inventory_snapshots")
df_prod = spark.table("mini_project_team_a3t7.silver.product_master")

# ── Step 2: Daily Sales Aggregation ───────────────────────

daily_sales = (
    df_pos
    .withWatermark("event_timestamp", "1 day")
    .withColumn("inventory_date", date_trunc("day", col("event_timestamp")))
    .groupBy("store_id", "product_id", "inventory_date")
    .agg(
        sum("quantity").alias("units_sold"),
        sum("total_amount").alias("revenue"),
        count("*").alias("transaction_count"),
        avg("unit_price").alias("avg_unit_price")
    )
)

# ── Step 3: Warehouse Aggregation ─────────────────────────

wh_by_product = (
    df_wh
    .groupBy("product_id")
    .agg(
        sum("current_stock_qty").alias("current_stock"),
        sum("available_stock_qty").alias("available_stock"),
        avg("reorder_level").alias("reorder_level"),
        avg("max_stock").alias("max_stock")
    )
)

# ── Step 4: Product Attributes ───────────────────────────

prod_slim = (
    df_prod.select(
        "product_id",
        "cost_price",
        "selling_price",
        "category",
        "supplier_id"
    )
)

# ── Step 5: Store Inventory Snapshot ─────────────────────

store_inv_slim = (
    df_sinv.select(
        "store_id",
        "product_id",
        col("inventory_quantity").alias("current_quantity"),
        "expiry_date"
    )
)

# ── Step 6: Build Streaming Fact ─────────────────────────

fact_stream = (
    daily_sales
    .join(wh_by_product, "product_id", "left")
    .join(prod_slim, "product_id", "left")
    .join(store_inv_slim, ["store_id", "product_id"], "left")

    .withColumn(
        "stock_value",
        round(
            coalesce(col("available_stock"), lit(0)) *
            coalesce(col("cost_price"), lit(0)), 2
        )
    )

    .withColumn(
        "days_on_hand",
        when(
            col("units_sold") > 0,
            round(
                coalesce(col("current_quantity"), lit(0)) /
                col("units_sold"), 1
            )
        ).otherwise(lit(999))
    )

    .withColumn(
        "reorder_quantity",
        greatest(
            lit(0),
            coalesce(col("reorder_level"), lit(0)) -
            coalesce(col("available_stock"), lit(0))
        )
    )

    .withColumn(
        "is_stockout",
        coalesce(col("current_quantity"), lit(0)) == 0
    )

    .withColumn(
        "is_low_stock",
        (coalesce(col("current_quantity"), lit(0)) > 0) &
        (coalesce(col("current_quantity"), lit(0)) <= 20)
    )
)

# ── Step 7: Upsert Function (Delta Merge) ─────────────────

def upsert_fact(batch_df, batch_id):

    if spark.catalog.tableExists(gold_table):

        delta_table = DeltaTable.forName(spark, gold_table)

        (
            delta_table.alias("t")
            .merge(
                batch_df.alias("s"),
                """
                t.store_id = s.store_id
                AND t.product_id = s.product_id
                AND t.inventory_date = s.inventory_date
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:

        (
            batch_df.write
            .format("delta")
            .partitionBy("inventory_date")
            .saveAsTable(gold_table)
        )

# ── Step 8: Write Streaming Output ───────────────────────

query = (
    fact_stream.writeStream
    .foreachBatch(upsert_fact)
    .outputMode("update")
    .option(
        "checkpointLocation",
        "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/daily_inventory_fact/"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# inventory_kpis1


from pyspark.sql.functions import (
    col, avg, sum, count, when, round,
    lit, coalesce, stddev, lag
)
from pyspark.sql.window import Window

# ── Step 1: Read from Gold fact table ─────────────────────────────────
fact = spark.table("mini_project_team_a3t7.gold.fact_inventory_full")

# ── Step 2: Derive stock_value upfront ────────────────────────────────
# fact_inventory_full does not store stock_value as a column
fact = fact.withColumn(
    "stock_value",
    col("current_stock_qty") * col("cost_price")
)

# ── Step 3: Window specs for rolling calculations ──────────────────────
w30 = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date")           # was inventory_date
             .rowsBetween(-30, 0))

w90 = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date")           # was inventory_date
             .rowsBetween(-90, 0))

# ── Step 4: Inventory KPIs ─────────────────────────────────────────────
inv_kpis = (fact

    # ── KPI 1: Inventory Turnover Ratio ───────────────────────────────
    # = COGS sold over 30 days / avg inventory value over 30 days
    # Higher = stock is moving fast (good)
    # Lower  = stock is sitting idle (capital locked up)
    .withColumn("cogs_30d",
        sum(col("units_sold") * col("cost_price")).over(w30))
    .withColumn("avg_stock_value_30d",
        avg("stock_value").over(w30))
    .withColumn("inventory_turnover_ratio",
        round(
            when(col("avg_stock_value_30d") > 0,
                 col("cogs_30d") / col("avg_stock_value_30d"))
            .otherwise(lit(0)),
            3
        )
    )

    # ── KPI 2: Overstock Risk Index ───────────────────────────────────
    # = current_stock_qty / (30-day avg daily sales × 30)
    # > 1.5 → holding 1.5x more stock than needed for 30 days (overstock)
    # < 0.5 → holding less than 15 days of stock (risk of stockout)
    # 999   → no sales in last 30 days (dead stock candidate)
    .withColumn("avg_daily_sales_30d",
        avg("units_sold").over(w30))
    .withColumn("overstock_risk_index",
        round(
            when(col("avg_daily_sales_30d") > 0,
                 col("current_stock_qty") /           # was current_stock
                 (col("avg_daily_sales_30d") * 30))
            .otherwise(lit(999)),
            2
        )
    )

    # ── KPI 3: Dead Stock Detection ───────────────────────────────────
    # Dead stock = zero units sold in last 90 days but stock still sitting
    # These products are costing storage space with zero revenue return
    .withColumn("units_sold_90d",
        sum("units_sold").over(w90))
    .withColumn("is_dead_stock",
        (col("units_sold_90d") == 0) & (col("current_stock_qty") > 0))  # was current_stock

    # ── KPI 4: Avg Days on Hand (30-day rolling) ──────────────────────
    # stock_cover_days is already computed in fact_inventory_full
    # (was days_on_hand in original — same concept, different name)
    # Filter out nulls (rows where units_sold = 0, cover = null)
    .withColumn("avg_days_on_hand_30d",
        round(
            avg(
                when(col("stock_cover_days").isNotNull(),
                     col("stock_cover_days"))          # was days_on_hand < 999
            ).over(w30),
            1
        )
    )

    # ── KPI 5: Stock Value at Risk ────────────────────────────────────
    # New KPI — value of inventory sitting in dead/overstock products
    # Helps quantify the financial impact of poor inventory management
    .withColumn("stock_value_at_risk",
        round(
            when(
                col("is_dead_stock") |
                (col("overstock_risk_index") > 1.5),
                col("stock_value")
            ).otherwise(lit(0)),
            2
        )
    )

    # ── Risk classification ───────────────────────────────────────────
    # Priority order: stockout > dead stock > overstock > low stock > healthy
    # stockout_flag == 1  replaces is_stockout boolean
    # reorder_flag == 1   replaces is_low_stock boolean
    .withColumn("inventory_health",
        when(col("stockout_flag") == 1,              "Critical — Stockout")   
       .when(col("is_dead_stock"),                   "Critical — Dead Stock")
       .when(col("overstock_risk_index") > 1.5,      "Warning — Overstock")
       .when(col("reorder_flag") == 1,               "Warning — Low Stock")   
       .otherwise("Healthy")
    )

    .select(
        # Keys
        "store_id",
        "product_id",
        "snapshot_date",               
        "category",
        "subcategory",
        "brand",

        # Rolling KPIs
        "inventory_turnover_ratio",
        "overstock_risk_index",
        "avg_daily_sales_30d",
        "avg_days_on_hand_30d",         
        "units_sold_90d",
        "cogs_30d",
        "avg_stock_value_30d",

        # Dead stock
        "is_dead_stock",

        # Stock columns (correct names from fact_inventory_full)
        "current_stock_qty",            
        "available_stock_qty",          
        "reserved_stock_qty",
        "stock_value",                  
        "stock_value_at_risk",          

        # Flags (integer 0/1 in fact_inventory_full, not booleans)
        "stockout_flag",               
        "reorder_flag",                 
        "overstock_flag",

        # Health label
        "inventory_health",
    )
)

# ── Step 5: Write to Gold ──────────────────────────────────────────────
(
    inv_kpis.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")           
    .saveAsTable("mini_project_team_a3t7.gold.inventory_kpis1")
)


In [0]:
# supplier_performance_fact

from pyspark.sql import functions as F

# 1. CONFIGURATION
SILVER_PO_TABLE = "mini_project_team_a3t7.silver.purchase_order"
GOLD_SUPPLIER_KPI_TABLE = "mini_project_team_a3t7.gold.supplier_performance_fact"
CHECKPOINT_SUPPLIER_KPI = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/supplier_kpis_stream"

# 2. READ STREAM FROM SILVER
df_po_stream = spark.readStream.table(SILVER_PO_TABLE)

# 3. ENRICH DATA (Fixing the Missing Column Error)
# We calculate the delay days before the aggregation happens
df_enriched = (
    df_po_stream
    .withColumn("delivery_delay_days", 
        F.datediff(F.col("actual_delivery_date"), F.col("expected_delivery_date")))
)

# 4. KPI AGGREGATIONS
supplier_kpis = (
    df_enriched
    .groupBy("supplier_id")
    .agg(
        # KPI 1: On-time Delivery %
        (F.sum(F.when(F.col("delivery_delay_days") <= 0, 1).otherwise(0)) / 
         F.count("*") * 100).alias("otd_pct"),

        # KPI 2: Lead Time Variance
        F.round(F.stddev("delivery_delay_days"), 2).alias("lead_time_variance"),

        # KPI 3: Average & Quality Metrics
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"),
        F.round(F.avg("quality_rating"), 2).alias("avg_quality_rating"),

        # Volume Metrics
        F.count("*").alias("total_orders"),
        F.round(F.sum(F.col("quantity_ordered") * F.col("unit_cost")), 2).alias("total_po_value")
    )
    
    # --- Composite Supplier Risk Score Logic ---
    .withColumn("norm_otd", F.col("otd_pct") / 100)
    .withColumn("norm_quality", F.coalesce(F.col("avg_quality_rating"), F.lit(0.0)) / 5.0)
    .withColumn("norm_lt_var", 
        F.when(F.col("lead_time_variance") > 0, F.lit(1.0) - (F.col("lead_time_variance") / 10.0))
         .otherwise(F.lit(1.0)))
    
    .withColumn("supplier_risk_score", 
        F.round(F.col("norm_otd") * 0.4 + F.col("norm_quality") * 0.3 + F.col("norm_lt_var") * 0.3, 3))
    
    .withColumn("actual_risk_tier",
        F.when(F.col("supplier_risk_score") > 0.55, "Low")
         .when(F.col("supplier_risk_score") > 0.53, "Medium")
         .otherwise("High"))
    
    .drop("norm_otd", "norm_quality", "norm_lt_var")
    .withColumn("_gold_processed_at", F.current_timestamp())
)

# 5. WRITE STREAM & DISPLAY
# 'complete' mode ensures the table always contains the most recent global summary
query = (supplier_kpis.writeStream
    .format("delta")
    .outputMode("complete") 
    .option("checkpointLocation", CHECKPOINT_SUPPLIER_KPI)
    .trigger(availableNow=True)
    .toTable(GOLD_SUPPLIER_KPI_TABLE))

query.awaitTermination()

In [0]:
# gold.demand_intelligence

from pyspark.sql.functions import (col, avg, sum, round, when, lit,
    month, year, date_format)
from pyspark.sql.window import Window

# ── Step 1: Read Gold fact table ───────────────────────────────────────
fact = spark.table("mini_project_team_a3t7.gold.daily_inventory_fact")

# ── Step 2: Define rolling windows ────────────────────────────────────
w7  = (Window.partitionBy("store_id", "product_id")
             .orderBy("inventory_date").rowsBetween(-7,  0))
w30 = (Window.partitionBy("store_id", "product_id")
             .orderBy("inventory_date").rowsBetween(-30, 0))
w90 = (Window.partitionBy("store_id", "product_id")
             .orderBy("inventory_date").rowsBetween(-90, 0))

# ── Step 3: Rolling averages ───────────────────────────────────────────
demand = (fact
    .withColumn("avg_units_7d",   round(avg("units_sold").over(w7),  2))
    .withColumn("avg_units_30d",  round(avg("units_sold").over(w30), 2))
    .withColumn("avg_units_90d",  round(avg("units_sold").over(w90), 2))
    .withColumn("avg_rev_7d",     round(avg("revenue").over(w7),    2))
    .withColumn("avg_rev_30d",    round(avg("revenue").over(w30),   2))

    # KPI: Seasonality Index = 7-day avg / 90-day avg
    # > 1.0 means current week is selling faster than the 90-day trend
    .withColumn("seasonality_index",
        round(when(col("avg_units_90d") > 0,
                   col("avg_units_7d") / col("avg_units_90d"))
              .otherwise(lit(1.0)), 3))

    # KPI: Trend Direction
    # Rising: selling 10%+ faster than last month
    # Falling: selling 10%+ slower than last month
    .withColumn("trend_direction",
        when(col("avg_units_7d") > col("avg_units_30d") * 1.1,  "Rising")
       .when(col("avg_units_7d") < col("avg_units_30d") * 0.9,  "Falling")
       .otherwise("Stable"))

    # Month-level seasonality (from your data: Dec & May are peak months)
    .withColumn("month_num",  month("inventory_date"))
    .withColumn("year_num",   year("inventory_date"))
    .withColumn("year_month", date_format("inventory_date", "yyyy-MM"))

    .select(
        "store_id", "product_id", "inventory_date",
        "year_num", "month_num", "year_month",
        "units_sold",                       # daily actual
        "avg_units_7d", "avg_units_30d", "avg_units_90d",
        "avg_rev_7d",   "avg_rev_30d",
        "seasonality_index", "trend_direction"
    )
)

# ── Step 4: Write to Gold ──────────────────────────────────────────────
(demand.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year_month")
    .saveAsTable("mini_project_team_a3t7.gold.demand_intelligence")
)


In [0]:
# fact_customer_sales

import pyspark.sql.functions as F

# Read the Silver Customer Master table
df_cust_silver = spark.table("mini_project_team_a3t7.silver.customers")

dim_customer = (
    df_cust_silver
    .select(
        "customer_id",
        "age_group",
        "gender",
        "zip_code",
        "loyalty_tier",
        "first_purchase_date",
        "preferred_categories"
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

# Write to Gold
dim_customer.write.format("delta").mode("append").saveAsTable("mini_project_team_a3t7.gold.dim_customer_segment_new")

df_pos_silver = spark.table("mini_project_team_a3t7.silver.pos_transactions")


dim_pos = (
    df_pos_silver
    .select(
        "transaction_id",
        "payment_method",
        "channel"
    )
    .dropDuplicates(["transaction_id"])
    .withColumn("_gold_processed_at", F.current_timestamp())
)

# Write to Gold
dim_pos.write.format("delta").mode("overwrite").saveAsTable("mini_project_team_a3t7.gold.dim_pos_transactions")

# Load the dimensions we just created
dim_cust = spark.table("mini_project_team_a3t7.gold.dim_customer_segment_new")
dim_pos_meta = spark.table("mini_project_team_a3t7.gold.dim_pos_transactions")
raw_txns = spark.table("mini_project_team_a3t7.silver.pos_transactions")

fact_sales = (
    raw_txns.alias("tx")
    # Join with Customer Dim to get Age/Loyalty
    .join(dim_cust.alias("c"), "customer_id", "inner")
    # Join with POS Dim to get Payment/Channel
    .join(dim_pos_meta.alias("p"), "transaction_id", "inner")
    
    .select(
        "tx.transaction_id",
        "tx.customer_id",
        "tx.store_id",
        "tx.product_id",
        "tx.event_timestamp",
        "tx.quantity",
        "tx.unit_price",
        "tx.total_amount",
        "c.age_group",
        "c.loyalty_tier",
        "p.channel",
        "p.payment_method"
    )
    .withColumn("inventory_date", F.to_date("event_timestamp"))
    .withColumn("_fact_processed_at", F.current_timestamp())
)

# Write to Gold (Partitioned by date for faster BI queries)
(fact_sales.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("inventory_date")
    .saveAsTable("mini_project_team_a3t7.gold.fact_customer_sales")
)